<a href="https://colab.research.google.com/github/Pazidu/Research-Project/blob/main/Notebooks/Two_stage_Dermomodel_III.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/drive')


Mounted at /drive


In [2]:
import os
import shutil
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping

print("TensorFlow:", tf.__version__)
print("GPU:", tf.test.gpu_device_name())

TensorFlow: 2.19.0
GPU: /device:GPU:0


In [3]:
BASE = "/content/newdata"
IMG_SRC = "/drive/MyDrive/Colab Notebooks/newdata"
CHECKPOINT_DIR = "/drive/MyDrive/checkpoints"

STAGE1_MODEL_PATH = "/drive/MyDrive/Models/stage1_model.keras"
FINAL_MODEL_PATH  = "/drive/MyDrive/Models/stage2_final.keras"

In [4]:
if os.path.exists(BASE):
    shutil.rmtree(BASE)

In [5]:
shutil.copytree(IMG_SRC, BASE)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [6]:
batch_size = 16
image_size = 300
FUSION_LAYER = "block4c_add"

In [7]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
])

In [8]:
def focal_loss(gamma=2., alpha=0.25):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        ce = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.pow(1 - y_pred, gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * ce, axis=1))
    return loss

In [9]:
def add_edge_map(image, label, training=False):
    image = tf.cast(image, tf.float32)

    # ✅ augmentation only for training
    if training:
        image = data_augmentation(image)

    image = preprocess_input(image)

    gray = tf.image.rgb_to_grayscale(image)

    # 🔥 multi-scale edge extraction
    e1 = tf.image.sobel_edges(gray)
    e2 = tf.image.sobel_edges(tf.image.resize(gray, [128,128]))

    e1 = tf.sqrt(tf.reduce_sum(tf.square(e1), axis=-1))
    e2 = tf.sqrt(tf.reduce_sum(tf.square(e2), axis=-1))

    e2 = tf.image.resize(e2, [image_size, image_size])

    edge = tf.concat([e1, e2], axis=-1)
    edge = edge / (tf.reduce_max(edge) + 1e-6)

    return (image, edge), label


def prepare_dataset(path, training):
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        image_size=(image_size, image_size),
        batch_size=batch_size,
        label_mode='categorical',
        shuffle=training
    )

    ds = ds.map(lambda x,y: add_edge_map(x,y,training),
                num_parallel_calls=tf.data.AUTOTUNE)

    return ds.prefetch(tf.data.AUTOTUNE)

In [10]:
train_ds = prepare_dataset(f"{BASE}/train", True)
val_ds   = prepare_dataset(f"{BASE}/valid", False)
test_ds  = prepare_dataset(f"{BASE}/test", False)

Found 8012 files belonging to 2 classes.
Found 1001 files belonging to 2 classes.
Found 1002 files belonging to 2 classes.


In [11]:
def channel_attention(x, ratio=8):
    c = x.shape[-1]

    gap = layers.GlobalAveragePooling2D()(x)
    d1 = layers.Dense(c // ratio, activation='relu')(gap)
    d2 = layers.Dense(c, activation='sigmoid')(d1)

    scale = layers.Reshape((1,1,c))(d2)
    return layers.Multiply()([x, scale])

In [12]:
def create_stage1_model():

    rgb_input = layers.Input(shape=(image_size, image_size, 3))
    edge_input = layers.Input(shape=(image_size, image_size, 2))  # 🔥 updated

    base = EfficientNetV2S(include_top=False, weights="imagenet", input_tensor=rgb_input)
    rgb_feat = base.get_layer(FUSION_LAYER).output

    x = layers.Conv2D(32,3,padding='same',activation='relu')(edge_input)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(64,3,padding='same',activation='relu')(x)
    x = layers.MaxPooling2D(2)(x)

    x = layers.Resizing(rgb_feat.shape[1], rgb_feat.shape[2])(x)

    rgb_feat = channel_attention(rgb_feat)
    x = channel_attention(x)

    fused = layers.Concatenate()([rgb_feat, x])

    fused = layers.Conv2D(256,3,padding='same',activation='relu')(fused)
    fused = layers.BatchNormalization()(fused)

    features = layers.GlobalAveragePooling2D(name="feature_layer")(fused)

    x = layers.BatchNormalization()(features)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(2, activation='softmax')(x)

    return tf.keras.Model(inputs=[rgb_input, edge_input], outputs=outputs)

In [13]:
lr_schedule = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2,
    min_lr=1e-5   # 🔥 fixed
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [14]:
print("\n🔥 Training Stage 1")

stage1_model = create_stage1_model()

stage1_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=focal_loss(),   # 🔥 improved
    metrics=["accuracy"]
)

checkpoint1 = ModelCheckpoint(
    f"{CHECKPOINT_DIR}/stage1_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

stage1_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[checkpoint1, lr_schedule, early_stop]
)

stage1_model.save(STAGE1_MODEL_PATH)



🔥 Training Stage 1
82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/20
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 842ms/step - accuracy: 0.5725 - loss: 0.1569
Epoch 1: val_accuracy improved from None to 0.67433, saving model to /drive/MyDrive/checkpoints/stage1_best.keras

Epoch 1: finished saving model to /drive/MyDrive/checkpoints/stage1_best.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 520s 913ms/step - accuracy: 0.6106 - loss: 0.1323 - val_accuracy: 0.6743 - val_loss: 0.0545 - learning_rate: 1.0000e-04
Epoch 2/20
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 686ms/step - accuracy: 0.7065 - loss: 0.0827
Epoch 2: val_accuracy improved from 0.67433 to 0.83017, saving model to /drive/MyDrive/checkpoints/stage1_best.keras

Epoch 2: finished saving model to /drive/MyDrive/checkpoints/stage1_best.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 370s 735ms/step - accuracy: 0.7267 - loss: 0.0784 - val_accuracy: 0.8302 - val_loss: 0.0347 - learning_rate: 1.0000e-04
Epoch 3/20
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 749ms/step - accuracy

In [15]:
print("\n🔥 Training Stage 2")

stage2_model = tf.keras.models.load_model(
    f"{CHECKPOINT_DIR}/stage1_best.keras",
    custom_objects={'loss': focal_loss()}
)

# 🔥 unfreeze ALL layers
for layer in stage2_model.layers:
    layer.trainable = True

stage2_model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-6),  # 🔥 better LR
    loss=focal_loss(),
    metrics=["accuracy"]
)

checkpoint2 = ModelCheckpoint(
    f"{CHECKPOINT_DIR}/stage2_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

stage2_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[checkpoint2, lr_schedule, early_stop]
)


🔥 Training Stage 2
Epoch 1/20
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 696ms/step - accuracy: 0.8105 - loss: 0.0504
Epoch 1: val_accuracy improved from None to 0.87812, saving model to /drive/MyDrive/checkpoints/stage2_best.keras

Epoch 1: finished saving model to /drive/MyDrive/checkpoints/stage2_best.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 428s 759ms/step - accuracy: 0.8093 - loss: 0.0522 - val_accuracy: 0.8781 - val_loss: 0.0232 - learning_rate: 3.0000e-06
Epoch 2/20
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 655ms/step - accuracy: 0.8189 - loss: 0.0461
Epoch 2: val_accuracy did not improve from 0.87812
501/501 ━━━━━━━━━━━━━━━━━━━━ 351s 699ms/step - accuracy: 0.8224 - loss: 0.0458 - val_accuracy: 0.8741 - val_loss: 0.0224 - learning_rate: 3.0000e-06
Epoch 3/20
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 658ms/step - accuracy: 0.8234 - loss: 0.0461
Epoch 3: val_accuracy did not improve from 0.87812
501/501 ━━━━━━━━━━━━━━━━━━━━ 352s 702ms/step - accuracy: 0.8225 - loss: 0.0474 - val_accuracy: 0.8631 - val_loss: 0.0229 -

In [16]:
def tta_evaluate(model, dataset):
    preds = []
    for (img, edge), y in dataset:
        p1 = model.predict([img, edge], verbose=0)
        p2 = model.predict([tf.image.flip_left_right(img), edge], verbose=0)
        preds.append((p1 + p2)/2)

    preds = tf.concat(preds, axis=0)
    true  = tf.concat([y for _,y in dataset], axis=0)

    acc = tf.keras.metrics.categorical_accuracy(true, preds)
    print("Final Test Accuracy (TTA):", tf.reduce_mean(acc).numpy())


In [17]:
best_model = tf.keras.models.load_model(
    f"{CHECKPOINT_DIR}/stage2_best.keras",
    custom_objects={'loss': focal_loss()}
)

tta_evaluate(best_model, test_ds)

stage2_model.save(FINAL_MODEL_PATH)

Final Test Accuracy (TTA): 0.88722557
